In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

df=pd.read_csv('datasets/train.csv')
df

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df['Date']=pd.to_datetime(df['Date'])
df.dtypes

In [ ]:
#sort data by time
df=df.sort_values('Date').reset_index(drop=True)
#loading the features dataset
features=pd.read_csv('datasets/features.csv')
features.head()

In [ ]:
features.info()

In [ ]:
#checking the null values in the data set of features
features.isnull().sum()

In [ ]:
#load stores data set
stores=pd.read_csv('datasets/stores.csv')
stores.head()

In [ ]:
stores.info()

In [ ]:
print(stores.isnull().sum())
print("-----------------------------")
print(stores.dtypes)

In [ ]:
#convert date column in features
features['Date']=pd.to_datetime(features['Date'])
features.dtypes

In [ ]:
print(df['IsHoliday'].dtype,features['IsHoliday'].dtype)
#sort features by date
features=features.sort_values('Date').reset_index(drop=True)


In [ ]:
#merging train with features using stores date and isholiday using left join
df_merged=pd.merge(df,features,how='left',on=['Store','Date','IsHoliday'])
print(df_merged.shape)
print("____________________________")
print(df.shape)
print("____________________________")
print(df_merged.head())
print('----------------------------------------')
print(df_merged.info())

In [ ]:
#merging df_merged with stores using left join
df_final=pd.merge(df_merged,stores,how='left',on='Store')
print(df_final.shape)
print(df_final.info())

In [ ]:
#saving the processed dataset for no loss
df_final.to_csv('walmart_sales_processed.csv',index=False)
df_final['Weekly_Sales'].describe()

In [ ]:
#sales by date joining
weekly_trend=(df_final.groupby('Date')['Weekly_Sales'].sum().reset_index())
#ploting the total walmart weekly sales
plt.figure(figsize=(12,5))
plt.plot(weekly_trend['Date'],weekly_trend['Weekly_Sales'])
plt.title('Total Weekly Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Weekly Sales')
plt.show()


In [ ]:
#comparing the average sales in hokidays
holiday_sales=(df_final.groupby('IsHoliday')['Weekly_Sales'].mean().reset_index())
holiday_sales

In [ ]:
#ploting the comparision using bar chart
plt.figure(figsize=(6,4))
plt.bar(holiday_sales['IsHoliday'].astype(str),holiday_sales['Weekly_Sales'])
plt.title('Average weekly sales: holiday vs non_holiday')
plt.xlabel('IsHoliday')
plt.ylabel('Average Weekly Sales')
plt.show()


In [ ]:
#average sales by store size
size_sales=(df_final.groupby('Size')['Weekly_Sales'].mean().reset_index())
size_sales.head()


In [ ]:
#ploting the size vs sales using scatter plot
plt.figure(figsize=(8,5))
plt.scatter(size_sales['Size'],size_sales['Weekly_Sales'])
plt.title('Store Size vs Average Weekly Sales')
plt.xlabel('Store Size')
plt.ylabel('Average Weekly Sales')
plt.show()

In [ ]:
type_sales=(df_final.groupby('Type')['Weekly_Sales'].mean().reset_index())
type_sales

In [ ]:
#ploting the type wise sales using barchart
plt.figure(figsize=(6,4))
plt.bar(type_sales['Type'], type_sales['Weekly_Sales'])
plt.title('Average Weekly Sales by Store Type')
plt.xlabel('Store Type')
plt.ylabel('Average Weekly Sales')
plt.show()

In [ ]:
#trying to do feature engineering


#making a dupilicate of df_final
df_fe=df_final.copy()

In [ ]:
df_fe['Year']=df_fe['Date'].dt.year
df_fe['Month']=df_fe['Date'].dt.month
df_fe['Week']=df_fe['Date'].dt.isocalendar().week.astype(int)

In [ ]:
df_fe[['Date','Year','Month','Week']].head()


In [ ]:
#models cant use text directly so changing a,b,c to 3,2,1
df_fe['Store_Type']=df_fe['Type'].map({'A':3,'B':2,'C':1})

In [ ]:
#log transform makes data more learnable and readable
df_fe['Log_Store_Size']=np.log1p(df_fe['Size'])

In [ ]:
#veriring the new columns we added
df_fe[['Type','Store_Type','Size','Log_Store_Size']].head()


In [ ]:
#using lag features for using past sales to predict future sales
df_fe=df_fe.sort_values(['Store','Dept','Date'])
df_fe['Sales_Lag_1']=(df_fe.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1))
df_fe[['Store','Dept','Date','Weekly_Sales','Sales_Lag_1']].head(10)



In [ ]:
df_fe['Sales_Lag_2']=(df_fe.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(2))
df_fe[['Store','Dept','Date','Weekly_Sales','Sales_Lag_1','Sales_Lag_2']].head(10)

In [ ]:
#using rolling average features to look at overall recent trend like average sales over last 3 weeks 
df_fe['Sales_Rolling_3']=(df_fe.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1).rolling(window=3).mean())
#shift(1) used to prevents data leakage
#(window=3) last 3 weeks average

In [ ]:
#using rolling average to last 5 week rolling average
df_fe['Sales_Rolling_5']=(df_fe.groupby(['Store', 'Dept'])['Weekly_Sales'].shift(1).rolling(window=5).mean())

In [ ]:
df_fe[['Store','Dept','Date','Weekly_Sales','Sales_Lag_1','Sales_Lag_2','Sales_Rolling_3','Sales_Rolling_5']].head(12)
df_fe.shape

In [ ]:
#droping rows with insufficient history filling them would create noise 
df_model=df_fe.dropna().reset_index(drop=True)


In [ ]:
df_model.isnull().sum

In [ ]:
#modeling 
#defining target and feature set
target='Weekly_Sales'
features = [
    'Year','Month','Week',
    'IsHoliday',
    'Temperature','Fuel_Price','CPI','Unemployment',
    'Store_Type','Log_Store_Size',
    'Sales_Lag_1','Sales_Lag_2',
    'Sales_Rolling_3','Sales_Rolling_5'
]

In [ ]:
#modeling
#model 1 linear regression for building a base line 

In [ ]:
x=df_model[features]
y=df_model[target]

#time-based train_test_split
split_date='2012-01-01'

x_train=x[df_model['Date']<split_date]
x_test=x[df_model['Date']>=split_date]

y_train=y[df_model['Date']<split_date]
y_test=y[df_model['Date']>=split_date]

In [ ]:
#scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
#train linear regression model
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(x_train_scaled,y_train)

In [ ]:
#making predictions
y_pred_lr=lr.predict(x_test_scaled)


In [ ]:
#evaluate model perfomance using rmse mae r2
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

rmse_lr=np.sqrt(mean_squared_error(y_test,y_pred_lr))
mae_lr=mean_absolute_error(y_test,y_pred_lr)
r2_lr=r2_score(y_test,y_pred_lr)

rmse_lr, mae_lr, r2_lr


In [ ]:
#lasso regression 
#train lasso with cross-validation
from sklearn.linear_model import LassoCV
lasso=LassoCV(cv=5,random_state=42)
lasso.fit(x_train_scaled,y_train)

In [ ]:
y_pred_lasso=lasso.predict(x_test_scaled)

In [ ]:
rmse_lasso=np.sqrt(mean_squared_error(y_test,y_pred_lasso))
mae_lasso=mean_absolute_error(y_test,y_pred_lasso)
r2_lasso=r2_score(y_test,y_pred_lasso)

rmse_lasso,mae_lasso,r2_lasso

In [ ]:
#model3
#using random forest 
from sklearn.ensemble import RandomForestRegressor
rf =RandomForestRegressor(n_estimators=300,max_depth=12,random_state=42,min_samples_leaf=5)
rf.fit(x_train, y_train)

In [ ]:
y_pred_rf=rf.predict(x_test)

In [ ]:
rmse_rf=np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf=mean_absolute_error(y_test, y_pred_rf)
r2_rf=r2_score(y_test, y_pred_rf)

rmse_rf,mae_rf,r2_rf

In [ ]:

feature_importance = pd.Series(rf.feature_importances_,index=features).sort_values(ascending=False)

feature_importance


In [ ]:
feature_importance.head(10).plot(kind='bar',title='Top Feature Importances')


In [ ]:
#trying ot save the predictig data as csv 
#Prepare latest data
latest_data=(df_model.sort_values('Date').groupby(['Store', 'Dept']).tail(1).reset_index(drop=True))
latest_data

In [ ]:
x_latest=latest_data[features]


In [ ]:
#prediction using random frorest that has be already trained
latest_data['Predicted_Weekly_Sales']=rf.predict(x_latest)
latest_data['Demand_Change']=(latest_data['Predicted_Weekly_Sales']-latest_data['Weekly_Sales'])
latest_data


In [ ]:
prediction_output = latest_data[
    [
        'Store',
        'Dept',
        'Date',
        'Weekly_Sales',
        'Predicted_Weekly_Sales',
        'Demand_Change'
    ]
]
prediction_output

In [ ]:
prediction_output.to_csv('walmart_next_week_sales_prediction.csv',index=False)


In [ ]:

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.metrics import explained_variance_score

evs = explained_variance_score(y_test, y_pred_rf)
r2 = r2_score(y_test, y_pred_rf)
mae = mean_absolute_error(y_test, y_pred_rf)
mse = mean_squared_error(y_test, y_pred_rf)
rmse = np.sqrt(mse)

In [ ]:
print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"Explained Variance: {evs:.4f}")